# SahayaVaani — Gnani Evon v3.3 vLLM GPU Serving Notebook

This notebook serves **Gnani AI's Evon v3.3** (open-weights 30B MoE model from Hugging Face `gnani/evon-v3.3`) using **vLLM** with an OpenAI-compatible API.

### Instructions:
1. In Google Colab menu, ensure Hardware accelerator is set to GPU: **Runtime > Change runtime type > GPU (T4 or A100)**.
2. Run all cells in sequence.
3. Copy the generated `VLLM_BASE_URL` tunnel link into your SahayaVaani `backend/.env`.
4. Your SahayaVaani voice pipeline is now powered by **Gnani Evon v3.3**.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install vLLM and pyngrok
!pip install -q vllm pyngrok

In [ ]:
# 3. Hugging Face Login (Optional if weights require acceptance)
from huggingface_hub import login

HF_TOKEN = ""  # Optional: paste your Hugging Face user token if required
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Authenticated with Hugging Face.")
else:
    print("Proceeding with public weights.")

In [ ]:
# 4. Launch vLLM OpenAI-Compatible Server for gnani/evon-v3.3 in background
import subprocess
import time

cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "gnani/evon-v3.3",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--max-model-len", "4096",
    "--trust-remote-code",
    "--dtype", "bfloat16"
]

print("Starting vLLM server for gnani/evon-v3.3 on port 8000...")
vllm_proc = subprocess.Popen(cmd)
time.sleep(25)
print("vLLM server initialized.")

In [ ]:
# 5. Expose via ngrok tunnel to get a public URL
from pyngrok import ngrok

# Enter your free ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"

if NGROK_TOKEN and NGROK_TOKEN != "YOUR_NGROK_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(8000)
    public_vllm_url = f"{tunnel.public_url}/v1"
    print("=" * 60)
    print("Gnani Evon v3.3 vLLM Public Endpoint:")
    print(public_vllm_url)
    print("\nAdd these lines to your SahayaVaani backend/.env:")
    print(f"VLLM_BASE_URL={public_vllm_url}")
    print("EVON_MODEL_NAME=gnani/evon-v3.3")
    print("EVON_ENABLED=true")
    print("=" * 60)
else:
    print("Please provide your NGROK_TOKEN in the cell above to generate a public URL.")

In [ ]:
# 6. Health check & test inference on local port
import requests

try:
    payload = {
        "model": "gnani/evon-v3.3",
        "messages": [
            {"role": "system", "content": "You are a bilingual Kannada healthcare assistant for hospital reception."},
            {"role": "user", "content": "ನಮಸ್ಕಾರ, ನನಗೆ ಕಾರ್ಡಿಯಾಲಜಿ ಅಪಾಯಿಂಟ್‌ಮೆಂಟ್ ಬೇಕು."}
        ],
        "max_tokens": 128,
        "temperature": 0.3
    }
    res = requests.post("http://localhost:8000/v1/chat/completions", json=payload, timeout=30)
    print("Evon v3.3 Output:")
    print(res.json()["choices"][0]["message"]["content"])
except Exception as e:
    print("Health check failed or server still loading weights:", e)